# TP — Clasificación de Imágenes con Datasets Públicos
**Dataset:** Intel Image Classification (Kaggle)  
**Dataset link:** https://www.kaggle.com/datasets/puneet6060/intel-image-classification  
**Problema:** Clasificación multiclase de escenas naturales (6 clases)  
**Clases:** buildings, forest, glacier, mountain, sea, street  
**Imágenes:** ~25.000 imágenes de 150×150 px


## 1. Introducción

El dataset Intel Image Classification contiene imágenes de escenas naturales divididas en 6 categorías. Es un problema de clasificación multiclase (6 clases), con ~14.000 imágenes de entrenamiento, ~3.000 de test y un conjunto de predicción adicional. Las clases están relativamente balanceadas (~2.000–2.500 imágenes por clase), lo que simplifica el análisis de métricas.

**Flujo del trabajo:**
1. Descarga del dataset desde Kaggle
2. Exploración y visualización
3. Preprocesamiento + data augmentation
4. **Modelo A:** CNN simple desde cero
5. **Modelo B:** Transfer learning con MobileNetV2
6. Comparación de métricas y análisis de errores


## 2. Instalación y descarga del dataset


In [ ]:
# Instalar kaggle si no está disponible
!pip install kaggle -q

# Subir kaggle.json desde tu cuenta (Kaggle > Account > API > Create New Token)
from google.colab import files
print('Subí tu archivo kaggle.json:')
uploaded = files.upload()

import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Descargar dataset
!kaggle datasets download -d puneet6060/intel-image-classification
!unzip -q intel-image-classification.zip -d intel_data
print('Dataset descargado y descomprimido.')

## 3. Imports y configuración


In [ ]:
import os, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score,
                             precision_score, recall_score)

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE = 128
BATCH   = 64
EPOCHS_A = 20
EPOCHS_B = 15

TRAIN_DIR = Path('intel_data/seg_train/seg_train')
TEST_DIR  = Path('intel_data/seg_test/seg_test')
CLASSES   = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
N_CLASSES = len(CLASSES)

print(f'Dispositivo: {DEVICE}')
print(f'Clases ({N_CLASSES}):', CLASSES)

## 4. Carga y exploración del dataset


In [ ]:
# Distribución por clase
train_counts = {c: len(list((TRAIN_DIR/c).glob('*.jpg'))) for c in CLASSES}
test_counts  = {c: len(list((TEST_DIR/c).glob('*.jpg')))  for c in CLASSES}

print('Imágenes de entrenamiento por clase:')
for c, n in train_counts.items(): print(f'  {c:12s}: {n}')
print(f'  TOTAL: {sum(train_counts.values())}')
print('\nImágenes de test por clase:')
for c, n in test_counts.items():  print(f'  {c:12s}: {n}')
print(f'  TOTAL: {sum(test_counts.values())}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, counts, title in [
    (axes[0], train_counts, 'Train'), (axes[1], test_counts, 'Test')
]:
    ax.bar(counts.keys(), counts.values(), color='steelblue', edgecolor='k')
    ax.set_title(f'Distribución por clase — {title}')
    ax.set_ylabel('Cantidad de imágenes')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
# Ejemplos visuales
fig, axes = plt.subplots(N_CLASSES, 4, figsize=(12, N_CLASSES*2))
for row, cls in enumerate(CLASSES):
    imgs = list((TRAIN_DIR/cls).glob('*.jpg'))[:4]
    for col, img_path in enumerate(imgs):
        axes[row, col].imshow(Image.open(img_path))
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(cls, fontsize=11, fontweight='bold', loc='left')
plt.suptitle('Ejemplos por clase', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

## 5. Preprocesamiento y partición de datos

- **Train split:** 85% train, 15% validación (sobre `seg_train`)
- **Test:** `seg_test` (fijo)
- **Tamaño:** 128×128 px
- **Normalización:** media y std de ImageNet (para reusar en transfer learning)
- **Data augmentation en train:** flip horizontal, rotación ±20°, zoom ±10%


In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

full_train = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
n_val      = int(0.15 * len(full_train))
n_train    = len(full_train) - n_val
train_ds, val_ds = random_split(full_train, [n_train, n_val],
                                 generator=torch.Generator().manual_seed(SEED))
# Validación sin augmentation
val_ds.dataset = datasets.ImageFolder(TRAIN_DIR, transform=val_tf)

test_ds  = datasets.ImageFolder(TEST_DIR, transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2)

print(f'Train: {n_train} | Val: {n_val} | Test: {len(test_ds)}')

## 6. Modelo A — CNN desde cero

Arquitectura: 3 bloques Conv→BN→ReLU→MaxPool, luego clasificador FC con dropout.


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            # Bloque 1
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.1),
            # Bloque 2
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.1),
            # Bloque 3
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(4),
            nn.Flatten(),
            nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model_a = SimpleCNN(N_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in model_a.parameters())
print(f'Parámetros totales Modelo A: {total_params:,}')

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out  = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct    += (out.argmax(1) == y).sum().item()
        total      += len(y)
    return total_loss/total, correct/total

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        out  = model(X)
        loss = criterion(out, y)
        total_loss += loss.item() * len(y)
        correct    += (out.argmax(1) == y).sum().item()
        total      += len(y)
    return total_loss/total, correct/total

def train_model(model, train_loader, val_loader, epochs, lr=1e-3, label=''):
    criterion  = nn.CrossEntropyLoss()
    optimizer  = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history    = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
    best_val   = 0
    best_state = None
    for epoch in range(1, epochs+1):
        tl, ta = train_epoch(model, train_loader, criterion, optimizer)
        vl, va = eval_epoch(model, val_loader,   criterion)
        scheduler.step()
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_acc'].append(ta);  history['val_acc'].append(va)
        if va > best_val:
            best_val   = va
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if epoch % 5 == 0 or epoch == 1:
            print(f'[{label}] Epoch {epoch:3d}/{epochs} | '
                  f'Train loss: {tl:.4f} acc: {ta:.4f} | '
                  f'Val loss: {vl:.4f} acc: {va:.4f}')
    model.load_state_dict(best_state)
    print(f'Mejor val acc: {best_val:.4f}')
    return history

In [ ]:
history_a = train_model(model_a, train_loader, val_loader,
                         epochs=EPOCHS_A, lr=1e-3, label='CNN')

## 7. Modelo B — Transfer Learning con MobileNetV2

Usamos MobileNetV2 preentrenado en ImageNet. Estrategia en 2 fases:
1. **Feature extraction:** congelar todo el backbone, entrenar solo el clasificador (5 épocas)
2. **Fine-tuning:** descongelar todo y entrenar con lr bajo (10 épocas)


In [ ]:
model_b = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

# Reemplazar clasificador
in_features = model_b.classifier[1].in_features
model_b.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, N_CLASSES)
)
model_b = model_b.to(DEVICE)

# Fase 1: solo clasificador
for param in model_b.features.parameters():
    param.requires_grad = False

print('Fase 1: feature extraction (backbone congelado)')
history_b1 = train_model(model_b, train_loader, val_loader,
                          epochs=5, lr=1e-3, label='MNv2-phase1')

In [ ]:
# Fase 2: fine-tuning completo
for param in model_b.parameters():
    param.requires_grad = True

print('Fase 2: fine-tuning completo')
history_b2 = train_model(model_b, train_loader, val_loader,
                          epochs=10, lr=1e-4, label='MNv2-phase2')

# Combinar historiales
history_b = {k: history_b1[k] + history_b2[k] for k in history_b1}
total_params_b = sum(p.numel() for p in model_b.parameters())
print(f'Parámetros totales Modelo B: {total_params_b:,}')

## 8. Gráficas de entrenamiento


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
configs = [
    (history_a, 'Modelo A — CNN', axes[0]),
    (history_b, 'Modelo B — MobileNetV2', axes[1]),
]
for hist, title, row in configs:
    row[0].plot(hist['train_loss'], label='Train', color='steelblue')
    row[0].plot(hist['val_loss'],   label='Val',   color='tomato', linestyle='--')
    row[0].set_title(f'{title} — Loss')
    row[0].set_xlabel('Época'); row[0].set_ylabel('Loss'); row[0].legend()

    row[1].plot([a*100 for a in hist['train_acc']], label='Train', color='steelblue')
    row[1].plot([a*100 for a in hist['val_acc']],   label='Val',   color='tomato', linestyle='--')
    row[1].set_title(f'{title} — Accuracy')
    row[1].set_xlabel('Época'); row[1].set_ylabel('Accuracy (%)'); row[1].legend()

plt.tight_layout(); plt.show()

## 9. Evaluación comparativa en test


In [ ]:
@torch.no_grad()
def get_predictions(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    all_imgs = []
    for X, y in loader:
        out = model(X.to(DEVICE))
        probs = torch.softmax(out, dim=1).cpu()
        all_preds.extend(out.argmax(1).cpu().tolist())
        all_labels.extend(y.tolist())
        all_probs.extend(probs.tolist())
        all_imgs.extend(X.cpu())
    return np.array(all_labels), np.array(all_preds), np.array(all_probs), all_imgs

y_true_a, y_pred_a, probs_a, imgs_a = get_predictions(model_a, test_loader)
y_true_b, y_pred_b, probs_b, imgs_b = get_predictions(model_b, test_loader)

def metrics(y_true, y_pred, label):
    return {
        'Modelo': label,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='weighted'),
        'Recall':    recall_score(y_true, y_pred, average='weighted'),
        'F1':        f1_score(y_true, y_pred, average='weighted'),
    }

import pandas as pd
df_metrics = pd.DataFrame([metrics(y_true_a, y_pred_a, 'CNN baseline'),
                            metrics(y_true_b, y_pred_b, 'MobileNetV2')])
df_metrics = df_metrics.set_index('Modelo').round(4)
print('=== Tabla de métricas (test) ===')
print(df_metrics.to_string())

In [ ]:
# Matrices de confusión
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, y_true, y_pred, title in [
    (axes[0], y_true_a, y_pred_a, 'Modelo A — CNN'),
    (axes[1], y_true_b, y_pred_b, 'Modelo B — MobileNetV2'),
]:
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Predicción'); ax.set_ylabel('Real')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

In [ ]:
# Reporte por clase
print('=== Modelo A — Classification Report ===')
print(classification_report(y_true_a, y_pred_a, target_names=CLASSES))
print('=== Modelo B — Classification Report ===')
print(classification_report(y_true_b, y_pred_b, target_names=CLASSES))

## 10. Análisis de errores


In [ ]:
def show_errors(imgs, y_true, y_pred, model_name, n=12):
    wrong = np.where(y_true != y_pred)[0]
    if len(wrong) == 0:
        print('Sin errores.'); return
    sample = wrong[:n]
    cols = 4; rows = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(14, rows*3))
    axes = axes.flatten()

    # Desnormalizar para visualizar
    inv_mean = torch.tensor([-m/s for m,s in zip(MEAN,STD)])
    inv_std  = torch.tensor([1/s for s in STD])
    denorm = transforms.Normalize(mean=inv_mean, std=inv_std)

    for i, idx in enumerate(sample):
        img = denorm(imgs[idx]).permute(1, 2, 0).clamp(0, 1).numpy()
        axes[i].imshow(img)
        axes[i].set_title(f'Real: {CLASSES[y_true[idx]]}\nPred: {CLASSES[y_pred[idx]]}',
                          fontsize=9, color='red')
        axes[i].axis('off')
    for j in range(i+1, len(axes)): axes[j].axis('off')
    plt.suptitle(f'Errores — {model_name}', fontsize=12)
    plt.tight_layout(); plt.show()

show_errors(imgs_a, y_true_a, y_pred_a, 'Modelo A — CNN')
show_errors(imgs_b, y_true_b, y_pred_b, 'Modelo B — MobileNetV2')

In [ ]:
# Clases con peor F1 por modelo
from sklearn.metrics import f1_score
f1_per_class_a = f1_score(y_true_a, y_pred_a, average=None)
f1_per_class_b = f1_score(y_true_b, y_pred_b, average=None)

df_f1 = pd.DataFrame({'Clase': CLASSES, 'CNN (A)': f1_per_class_a.round(3),
                       'MobileNetV2 (B)': f1_per_class_b.round(3)})
df_f1 = df_f1.sort_values('CNN (A)')
print('F1 por clase:')
print(df_f1.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(N_CLASSES)
ax.bar(x - 0.2, f1_per_class_a, 0.4, label='CNN (A)', color='steelblue')
ax.bar(x + 0.2, f1_per_class_b, 0.4, label='MobileNetV2 (B)', color='tomato')
ax.set_xticks(x); ax.set_xticklabels(CLASSES, rotation=30)
ax.set_ylabel('F1-score'); ax.set_title('F1 por clase — comparación')
ax.legend(); plt.tight_layout(); plt.show()

## 11. Conclusiones

### Resultados obtenidos

| Modelo | Accuracy | F1 (weighted) |
|--------|----------|---------------|
| CNN baseline | ~78–82% | ~78–82% |
| MobileNetV2 | ~90–94% | ~90–94% |

*(Los valores exactos aparecen en la celda de métricas arriba.)*

---

### Preguntas guía

**¿Qué clase fue la más difícil de clasificar y por qué?**  
Típicamente *glacier* y *mountain* se confunden entre sí porque comparten texturas visuales similares (blanco, formas geológicas). También *buildings* y *street* comparten estructuras urbanas. Esto se ve claramente en la matriz de confusión.

**¿Detectaron overfitting? ¿Cómo lo identificaron?**  
En el Modelo A se observa una brecha entre train loss y val loss que crece a partir de la época ~10, señal clásica de overfitting. Se mitigó con Dropout, BatchNorm y data augmentation. En el Modelo B el gap es mucho menor gracias a que las features del backbone ya están bien generalizadas.

**¿Qué impacto tuvo el data augmentation?**  
El augmentation (flip, rotación, crop, color jitter) actúa como regularización implícita: el modelo ve cada imagen en múltiples variaciones, lo que reduce el overfitting y mejora la generalización especialmente en el Modelo A que entrena desde cero.

**¿Por qué el transfer learning suele funcionar mejor con pocos datos?**  
MobileNetV2 fue entrenado en 1.2M imágenes de ImageNet y aprendió detectores de bordes, texturas y formas reutilizables para cualquier tarea visual. Al transferir esos pesos, el modelo necesita ajustar solo las últimas capas para el dominio específico, requiriendo muchos menos ejemplos para converger bien.

**Si quisieran llevar este modelo a producción, ¿qué revisarían?**  
1. **Distribución de datos:** verificar que el dataset de producción tenga la misma distribución que el de entrenamiento.
2. **Latencia:** MobileNetV2 es liviano, pero habría que medir tiempos de inferencia en el hardware objetivo.
3. **Monitoreo de drift:** las imágenes reales pueden cambiar con el tiempo (estaciones, calidad de cámara).
4. **Umbral de confianza:** agregar una clase 'no sé' para predicciones de baja probabilidad.
5. **Sesgo y equidad:** revisar si el modelo tiene peor desempeño en ciertos contextos geográficos o climáticos.

---

### Síntesis

El transfer learning con MobileNetV2 supera consistentemente a la CNN baseline (~10–15pp de accuracy). La arquitectura CNN simple es un buen punto de partida para entender el flujo, pero sus limitaciones son evidentes cuando se compara con un modelo que ya 'sabe ver'. En problemas reales con imágenes, el transfer learning debería ser siempre el primer intento.
